# MORQEX geometry workflow / MORQEX几何建模

256³ original voxels, live2D/3D and distinct voxel-reference, raw-TetGen and GEMsim-adaptive workflows. Execute a copy outside source.
原始256³体素、实时二维/三维及独立的体素参考、原始TetGen、GEMsim自适应网格；请在源码外执行副本。

In [ ]:
from pathlib import Path
import sys,os
SOURCE=Path('/home/zephyrain/lw/MORQEX')
sys.path.insert(0,str(SOURCE/'src'))
WORK=Path('/home/zephyrain/lw/morph_workspace/microelectrode_gallery/notebook_example')
WORK.mkdir(parents=True,exist_ok=True)
import numpy as np
from morqex.modeling import from_regions,Shape,from_shapes


## Live view / 实时预览
The same display handle is updated during construction.
生成过程复用同一显示句柄。

In [ ]:
blank=from_regions(np.zeros((256,)*3,np.uint8),spacing=(0.5e-6,)*3)
view=blank.show(maximum_3d_size=40,height=680,title='Live geometry / 实时几何')


In [ ]:
rng=np.random.default_rng(20261127)
objects=[]
for k in range(18):
    angle=float(rng.uniform(-0.8,0.8))
    R=np.array([[np.cos(angle),0,np.sin(angle)],[0,1,0],[-np.sin(angle),0,np.cos(angle)]])
    objects.append(Shape('capsule',1,tuple(rng.uniform(15e-6,113e-6,3)),(4e-6,4e-6,float(rng.uniform(12e-6,30e-6))),tuple(map(tuple,R)),0.2))
model=from_shapes(objects,shape=(256,)*3,spacing=(0.5e-6,)*3,progress=view.update,progress_every=6)
view.update(model)
view.save_html(WORK/'live_geometry.html')


## Full tetrahedral mesh / 完整体四面体网格
The full256³ array is meshed; displayLOD does not enter the mesh calculation.
对完整256³数组生成网格，预览LOD不参与计算。

In [ ]:
target=WORK/'mesh.h5'
if not target.exists():
    artifact=model.mesh_to(target,regions=[1],slab_depth=2)
    print(artifact)
else:
    print('Existing mesh retained / 保留已有网格:',target)


In [ ]:
model.save(WORK/'voxels.h5')
print('HTML:',WORK/'live_geometry.html')
print('ParaView:',WORK/'mesh.xdmf')


In [ ]:
from morqex.modeling import MeshArtifact
import h5py
with h5py.File(WORK/'mesh.h5') as f:
    saved=MeshArtifact(WORK/'mesh.h5',WORK/'mesh.xdmf',int(f.attrs['node_count']),int(f.attrs['tetrahedra']),(1,))
mesh_view=saved.show(first_cell=0,cell_count=600,height=620)
mesh_view.save_html(WORK/'actual_mesh.html')


## Native TetGen + COMSOL / 原生 TetGen 与 COMSOL

This is a separate24³ multiphase example, **not** a replacement for the preceding full256³ voxel mesh.
此处是独立24³多相示例，不冒充前面完整256³体素网格。
Compile once using `MORQEX_BUILD_TETGEN=1 pip install .`, or the explicit external build shown below.
先用可选 pip 编译，或显式执行下面的外部构建；不在源码内存放二进制。
See [the bilingual tutorial](../05_tetgen_and_comsol.md) for all mesh routes and quality limits.


In [ ]:
from morqex.modeling import (build_tetgen,from_toml,mesh_from_toml,export_comsol,read_comsol)
from morqex.modeling.tetgen_backend import library_path
from datetime import datetime,timezone
os.environ.setdefault('MORQEX_WORKSPACE',str(WORK.parent))
try:
    print('Native library:',library_path())
except RuntimeError:
    print('Explicit one-job external build:',build_tetgen(WORK.parent))
recipe=SOURCE/'configs/geometry_tetgen.toml'
native_model=from_toml(recipe)
native_mesh=mesh_from_toml(native_model,recipe)
print(native_mesh.report)
assert native_mesh.signed_volumes().min()>0


In [ ]:
from IPython.display import display
native_dir=WORK/'tetgen'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
native_dir.mkdir(parents=True)
display(native_model.viewer(maximum_3d_size=32,title='24³ source regions / 原始分区'))
display(native_mesh.viewer(cell_count=2000,title='Actual native TetGen cells / 真实 TetGen 单元'))
native_model.viewer(maximum_3d_size=32).save_html(native_dir/'voxels.html')
native_mesh.viewer(cell_count=2000).save_html(native_dir/'mesh.html')
native_artifact=native_mesh.save_hdf5(native_dir/'mesh.h5')
comsol_file=export_comsol(native_mesh,native_dir/'mesh.mphtxt',region_names=native_model.region_names)
back=read_comsol(comsol_file)
np.testing.assert_array_equal(back['points'],native_mesh.points)
np.testing.assert_array_equal(back['elements']['tet']['cells'],native_mesh.cells)
print('COMSOL format/numerical read-back passed / 格式与数值读回通过:',comsol_file)
print('Actual COMSOL GUI import was not run / 未执行实际 COMSOL GUI 导入')


## GEMsim weighted ACVD + true volume sections / 加权自适应曲面与真实体切面

This separate full256³ example uses the new resolved-feature recipe and original weighted ACVD kernel; the TOML file supplies meshing options only.
此处是独立完整256³可执行示例，通过API明确指定几何；TOML只提供网格选项。完整十种256³案例由批量图库脚本生成。
The earlier voxel-split and raw-TetGen examples are retained as different methods.
前面的体素剖分及原始TetGen示例保留，方法之间不混淆。


In [ ]:
from morqex.modeling import build_acvd,AdaptiveMeshOptions,resolved_electrode
from morqex.modeling.acvd_backend import load_acvd
try:
    print('Weighted ACVD:',load_acvd().__file__)
except RuntimeError:
    # Explicit build requires Cython / 显式源码编译需要Cython
    print('Explicit single-job build:',build_acvd(WORK.parent))
adaptive_recipe=SOURCE/'configs/geometry_adaptive.toml'
adaptive_model=resolved_electrode('granular',size=256,spacing=0.5e-6,seed=20261127,cbd_thickness_voxels=4,minimum_feature_voxels=4)
adaptive_result=mesh_from_toml(adaptive_model,adaptive_recipe)
print(adaptive_result.report)
assert set(adaptive_result.phases)=={0,1,2}
assert adaptive_result.signed_volumes().min()>0


In [ ]:
adaptive_dir=WORK/'adaptive'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
adaptive_dir.mkdir(parents=True)
display(adaptive_model.viewer(maximum_3d_size=40,title='Source active/CBD/pore / 原始三相结构'))
display(adaptive_result.adaptive_viewer(title='Weighted ACVD surface + real volume sections / 自适应曲面与真实体切面'))
adaptive_model.save(adaptive_dir/'geometry.h5')
adaptive_result.save_hdf5(adaptive_dir/'mesh.h5')
adaptive_result.adaptive_viewer().save_html(adaptive_dir/'adaptive.html')
export_comsol(adaptive_result,adaptive_dir/'mesh.mphtxt',region_names=adaptive_model.region_names)
print('Actual adaptive artifacts / 自适应结果:',adaptive_dir)
print('Quality requests met:',adaptive_result.report['quality_limits_satisfied'])
